# Lab 11 — Observability, Tracing & Debugging

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

**Learning objectives** — after this lab you can:

- explain why classical debugging (reproduce, breakpoint, stack trace) fails on nondeterministic agents, and why **there is no stack trace for a bad decision**,
- build a **plain-Python tracer**: a `Tracer` that emits a tree of timed, attributed **spans** (one **trace** per run, `llm_call` and tool spans nested under `step` spans),
- **instrument the tool boundary** with a `traced(...)` wrapper so every tool and LLM call records its inputs, outputs, token counts, latency, and errors,
- **persist** traces as **JSONL** and **render** the span tree and a token/cost account,
- **diagnose** a prepared faulty run *from its trace alone*: spot the **loop signature**, zoom into payloads, and name the root cause (a **tool-contract bug**, not a model bug),
- **fix and prove**: repair the tool contract and build a **replay-based regression test** with a **trajectory-property** assertion,
- add a **PII scrubber** to the export path and verify it with a seeded email address.

> ⏱️ Estimated time: 90–120 minutes. Everything here is **offline and synthetic**. The lecture's
> lab uses a local Langfuse container; here we **reimplement the same trace/span model in plain
> Python** so the whole lab runs without any backend. Ollama is used only for one *optional*
> end-to-end cell in Part G — every core cell runs without it.

## Theory recap — seeing what a nondeterministic agent did

### There is no stack trace for a bad decision

Last week (S10) we *bounded* what the agent can do — sandboxes below, guardrails above. But the
fence is **opaque**: a contained run can still fail uselessly (loop, burn budget, mislead), and
the guardrail tells you only that a limit was reached, not *why*. Classical debugging cannot close
that gap, for **structural** reasons:

- **No reproduction** — behaviour is sampled from a distribution; a rerun draws a *new trajectory*. Temperature zero does not rescue you (providers rarely guarantee bitwise determinism, and the input — live tool results, timestamps, retrieved content — is rarely identical).
- **No stack trace** — the worst failures throw nothing. Choosing a wrong tool, trusting a junk source, paraphrasing a query instead of giving up — each is a *successful function call*. The defect is **semantic**.
- **No breakpoints** — the decision happens in a forward pass through the weights; there is no line where the agent "decides".
- **Delayed surfacing** — a misread tool result at step 3 quietly poisons the context; the visible nonsense appears at step 14.

The consequence: **debugging becomes post-hoc reconstruction from recorded evidence.** If the
evidence was not recorded at runtime, it is *gone* — the process memory is released and the next
run samples a different trajectory.

### Traces and spans

The fix is borrowed from distributed systems (Dapper, Sigelman et al., 2010) and adapted to
agents:

- A **trace** is the complete record of *one run*, identified by a single **trace ID**.
- A **span** is one timed operation: a `name`, `start`/`end` timestamps, a `parent`, and a bag of
  key-value **attributes**. Spans **nest** via parent links, so a trace is a **tree** — and the
  tree structure is not decoration: **parent–child links encode causality** (which tool result fed
  which decision).
- The agent loop maps onto the tree: a **root span** for the run, a **step span** per loop
  iteration, and inside each step the two span kinds that matter most — the **LLM span** (exact
  prompt, exact completion, model ID, token counts) and the **tool span** (name, exact arguments,
  exact result).

### What every span must carry

Unlike classical tracing, agent spans record **full inputs and outputs** — the exact prompt and
completion, the exact tool arguments and result — because *the model's decision is a function of
precisely what it saw*. Counts tell you *that* something went wrong; only payloads tell you *what*
the model actually saw and said. Each span also carries **token accounting** (input, output,
cached, reasoning — kept separate, because the growing context is re-sent every step and dominates
loop cost), **cost** (tokens × price, attributed per run/step/tool), **latency** (wall-clock per
span), and **metadata** (model ID, sampling parameters, prompt version, errors, retries).

### The three pillars, and why traces lead

**Logs** (timestamped structured events), **traces** (the causal span tree per run), **metrics**
(aggregates across runs). For agents, invert the classical hierarchy: **the trace is the primary
artifact**, because agent debugging is about individual *semantic* failures that live only in
payloads — and from a rich span tree, logs (spans as events) and metrics (summed counts,
latencies) are *derivable*. Get tracing right first; the other pillars come along for free.

### Reading a trace, then replaying it

The method is **shape → zoom → root cause**: read the tree's shape first (a uniform repeated block
is a **loop signature**), then zoom into the implicated spans' payloads, then walk back to the
first anomalous step. A recorded trace freezes one nondeterministic run into a **deterministic**
artifact, which makes **replay** possible (re-run the loop, serving tool results from the
recording — isolating the *decisions*) and turns yesterday's diagnosed failure into tonight's
**regression test**. Assert **trajectory properties** ("no identical tool call repeated more than
twice"; "every claim cites a fetched source"), not exact paths — and remember one passing replay
proves little for a stochastic system (sample several runs; that is S14).

### This lab

You will give your S10 research agent a **flight recorder**: a plain-Python tracer, a `traced`
wrapper at the tool boundary, JSONL persistence, tree/token rendering, a diagnosis of a prepared
nonterminating run, a **replay regression test**, and a **PII scrubber** on the export path —
exactly the arc the lecture announced (Slide 23).

## Part A — Setup & the offline corpus

Almost nothing here needs an LLM: the tracer and the whole debugging workflow are **plain Python**
and run offline. Ollama is used only for one *optional* end-to-end cell in Part G, so the
connectivity check below is friendly and non-blocking.

The build created a tiny corpus in `data/`:

- `fetched_pages.json` — three offline "web" pages on two topics (`tracing`, `loops`). A third
  topic, `obscure_topic`, resolves to **nothing** — the planted no-results case from the lecture.
- `bad_run_trace.jsonl` — a **pre-recorded faulty trace** of the run that never ends, one span per
  line. In Part E you will diagnose it *without reading any tool source code* — exactly as you
  would in production, where all you have is the recording.

In [ ]:
import os
import re
import json
import time
import uuid
from dataclasses import dataclass, field, asdict
from typing import Any, Optional

import pandas as pd

# --- friendly, non-blocking Ollama check (only Part G's optional cell needs it) ---
MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:7b")
OLLAMA_OK = False
try:
    import ollama
    ollama.list()
    OLLAMA_OK = True
    print(f"Ollama reachable — the optional end-to-end cell (Part G) can use {MODEL!r}.")
except Exception as exc:
    print("Ollama not reachable:", exc)
    print("→ That is fine: the tracer and the whole debugging workflow run offline.")
    print("  For the one optional LLM cell, start Ollama with `ollama serve` and")
    print("  `ollama pull qwen2.5:7b`.")

# --- load the offline corpus ---
with open("data/fetched_pages.json", "r", encoding="utf-8") as f:
    PAGES = json.load(f)
PAGES_BY_TOPIC = {}
for p in PAGES:
    PAGES_BY_TOPIC.setdefault(p["topic"], []).append(p)
print(f"\nLoaded {len(PAGES)} offline pages on topics:", sorted(PAGES_BY_TOPIC))
print("Note: the topic 'obscure_topic' resolves to NO pages — the planted no-results case.")

> **Q:** Explain why "we reran the failing task and it worked" is almost no evidence that an agent bug is fixed.
<details><summary>Click for answer</summary>

Agent behaviour is sampled from a probability distribution on each run, so a rerun draws a new
trajectory rather than reproducing the old one. A passing rerun shows only that *at least one*
successful trajectory exists — which was never in doubt; the failing trajectory remains in the
distribution with unchanged probability. Verifying a fix requires replaying the recorded failing
conditions, or sampling many runs and comparing failure rates before and after.
</details>

## Part B — A plain-Python tracer (spans as a tree)

The lecture uses Langfuse; the **data model** is what matters, and it is small enough to build
ourselves. A `Span` is one timed operation with a `name`, a `kind`, `start`/`end` timestamps, a
`parent_id`, and an **attributes** dict for payloads. A `Tracer` owns one **trace ID**, hands out
spans, and stores them flat (the parent links reconstruct the tree on demand).

Complete the two gaps: record the **end time / latency** when a span closes, and give each new
span the **current open span as its parent** so the tree is built correctly.

In [ ]:
@dataclass
class Span:
    span_id: str
    parent_id: Optional[str]
    trace_id: str
    name: str
    kind: str                      # "run" | "step" | "llm" | "tool"
    start_ms: float
    end_ms: Optional[float] = None
    attributes: dict = field(default_factory=dict)

    def set(self, **kw):
        """Attach payloads/metadata to the span (chainable)."""
        self.attributes.update(kw)
        return self

    @property
    def latency_ms(self):
        return None if self.end_ms is None else self.end_ms - self.start_ms


class Tracer:
    """Owns one trace (one run). Spans nest via a stack of currently-open spans."""

    def __init__(self):
        self.trace_id = "trace-" + uuid.uuid4().hex[:8]
        self.spans: list[Span] = []
        self._stack: list[str] = []          # ids of currently-open spans
        self._t0 = time.time()

    def _now_ms(self):
        return 1000.0 * (time.time() - self._t0)

    def start(self, name, kind, **attributes):
        span = Span(
            span_id="s" + uuid.uuid4().hex[:8],
            parent_id=self._stack[-1] if self._stack else None,   # current open span is the parent
            trace_id=self.trace_id,
            name=name,
            kind=kind,
            start_ms=self._now_ms(),
            attributes=dict(attributes),
        )
        self.spans.append(span)
        self._stack.append(span.span_id)
        return span

    def end(self, span):
        # GAP: record the end time so latency_ms is defined, then pop the stack.
        span.end_ms = ___
        # pop this span off the open-span stack (it is the most recent)
        if self._stack and self._stack[-1] == span.span_id:
            self._stack.pop()
        return span


# smoke test: a run span with one nested step
tr = Tracer()
run = tr.start("agent_run", "run", task="demo")
step = tr.start("step_0", "step", index=0)
tr.end(step)
tr.end(run)
print("trace id:", tr.trace_id, "| spans:", len(tr.spans))
print("step parent is run:", tr.spans[1].parent_id == run.span_id)
print("run latency recorded:", run.latency_ms is not None)

<details>
<summary><b>Click here for the solution</b></summary>

```python
span.end_ms = self._now_ms()
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

`start(...)` sets `parent_id` to the currently-open span (the top of `self._stack`) **before**
pushing the new span, so spans opened inside another span become its children — that stack is
exactly what turns a flat list into a tree. `end(...)` stamps `end_ms` from the same monotonic
clock (`_now_ms`), which makes `latency_ms` well-defined, and then pops the span so the *next*
`start` attaches to the correct parent. Recording only on close (rather than never) is the whole
point of the lecture's `finally` block: a span you forget to end has no latency and no export.
</details>

> **Q:** Why is the *parent–child* structure of spans essential rather than cosmetic?
<details><summary>Click for answer</summary>

Parent–child links record causality: which loop step contained which LLM call, and which tool
result was available when which decision was made. Debugging questions are causal questions —
"what did the model see before it chose this?" — and a flat event list forces you to reconstruct
that ordering heuristically from timestamps, which fails under concurrency. The tree gives the
answer structurally.
</details>

## Part C — Instrumenting the tool boundary

The lecture's key move (Slide 16): **one wrapper makes every tool call a recorded span**. We write
a `traced(...)` decorator that, given the active tracer, wraps a tool so that each call opens a
tool span with the **verbatim arguments**, records the **result** on success or the **error** on
failure, and — in a `finally` block — always **closes** the span (timing and export happen on
*every* path, including failures).

Two rules from the lecture, both load-bearing:

1. **Errors are recorded, then re-raised unchanged** — recording must never alter behaviour.
2. **The `finally` block always ends the span** — a span recorded only on the happy path is
   worthless, because the unhappy path is the one you will be debugging.

Complete the gaps: record the `result` on success, and re-raise so behaviour is unchanged.

In [ ]:
def traced(tracer, tool):
    """Wrap `tool` so every call becomes a recorded tool span on the given tracer."""
    def run(**args):
        span = tracer.start(tool.__name__, "tool", args=args)
        try:
            out = tool(**args)
            span.set(result=out, error=None)      # record the verbatim result
            return ___                             # GAP: return the tool's output unchanged
        except Exception as e:
            span.set(error=str(e))
            ___                                    # GAP: re-raise so behaviour is unchanged
        finally:
            tracer.end(span)                       # ALWAYS close: timing + export on every path
    return run


# --- the toy research tools (offline) ---
def search_web(query):
    """BUGGY CONTRACT (v1): returns a list of hits, or [] for no results — [] looks like success."""
    topic = query.strip().lower().split()[0] if query.strip() else ""
    hits = PAGES_BY_TOPIC.get(topic, [])
    return [{"url": p["url"], "title": p["title"]} for p in hits]   # [] when the topic is unknown


def fetch_page(url):
    for p in PAGES:
        if p["url"] == url:
            return p["body"]
    raise ValueError(f"unknown url: {url}")


# smoke test: wrap the tools on a fresh tracer and make two calls
tr = Tracer()
run = tr.start("agent_run", "run", task="smoke")
s_search = traced(tr, search_web)
s_fetch = traced(tr, fetch_page)
print("hits for 'tracing':", s_search(query="tracing"))
print("hits for 'obscure_topic':", s_search(query="obscure_topic"), "  <- empty == 'success'")
tr.end(run)
print("tool spans recorded:", sum(1 for s in tr.spans if s.kind == "tool"))

<details>
<summary><b>Click here for the solution</b></summary>

```python
out = tool(**args)
span.set(result=out, error=None)
return out          # return the tool's output unchanged
except Exception as e:
    span.set(error=str(e))
    raise           # re-raise so behaviour is unchanged
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

`traced` sits at the **tool boundary**, outside the model — like the S10 policy wrapper, nothing
in the context can talk it out of its job. On success it records the exact result and returns it
*unchanged*, so the caller sees precisely what it would have seen untraced. On failure it records
the error message and then `raise` re-raises the *same* exception, so the loop's error handling is
unaffected — instrumentation must be observationally transparent. The `finally` block runs on all
three exits (return, raise, even a cancellation), guaranteeing the span is timed and closed:
recording the unhappy path is the entire reason the wrapper exists. The LLM client would get the
same treatment with different attributes (prompt, completion, token counts, model ID).
</details>

Now the **instrumented agent loop**. A tiny scripted "planner" stands in for the
model (deterministic, so the lab is reproducible), but every LLM decision and every tool call is
recorded exactly as a real loop would record them. Each iteration opens a **step** span; inside it
an **llm** span (prompt/completion/tokens) and then a **tool** span. The loop stops on success or
when the **step budget** (S10) fires — the containment that turns an infinite loop into a finite,
*recorded* one.

In [ ]:
def run_research_agent(task_topic, search_tool, max_steps=20):
    """A scripted, fully-instrumented loop. Returns (tracer, outcome)."""
    tr = Tracer()
    run = tr.start("agent_run", "run", task=f"Write a short report on {task_topic!r}.")
    context_tokens = 200                      # grows every step (context is re-sent)
    outcome = None

    for step in range(max_steps + 1):
        if step > max_steps:                  # S10 step budget: contain the loop
            outcome = "KILLED: step budget exceeded"
            break

        step_span = tr.start(f"step_{step}", "step", index=step)

        # --- llm span: the "planner" decides to search (paraphrasing if it saw nothing) ---
        query = task_topic if step == 0 else f"{task_topic} rephrased v{step}"
        llm = tr.start("llm_call", "llm")
        context_tokens += 40                  # the growing context is re-sent every step
        llm.set(model=MODEL, prompt=f"...context... search for: {query}",
                completion=f'{{"tool":"search_web","args":{{"query":"{query}"}}}}',
                input_tokens=context_tokens, output_tokens=18)
        tr.end(llm)

        # --- tool span: run the (possibly buggy) search tool through `traced` ---
        traced_search = traced(tr, search_tool)
        try:
            hits = traced_search(query=query)
        except StopIteration as e:            # the FIXED tool signals "no results" explicitly
            outcome = f"REPORTED: no sources found ({e})"
            tr.end(step_span)
            break

        if hits:                              # success path: we found something → write & stop
            outcome = f"OK: report drafted from {len(hits)} source(s)"
            tr.end(step_span)
            break

        # empty hits + buggy contract → the planner will just paraphrase and loop
        tr.end(step_span)

    run.set(outcome=outcome, steps=step)
    tr.end(run)
    return tr, outcome


# a HEALTHY run first (topic 'tracing' has pages) — terminates fast
tr_ok, outcome_ok = run_research_agent("tracing", search_web)
print("healthy run:", outcome_ok, "| steps:", tr_ok.spans[0].attributes["steps"])

<details>
<summary><b>Click here for the solution</b></summary>

This cell has **no gap** — it is provided complete so you can focus on *reading* the traces it
produces. Note the three span kinds nested per iteration (`step` → `llm`, `tool`), the growing
`input_tokens` (the context is re-sent every step), and the two clean exits: `hits` found (draft &
stop) or the S10 step budget firing. The `StopIteration` branch is dormant now — it becomes live
in Part F once the tool contract is *fixed* to signal "no results" explicitly.
</details>

### Break it: the planted no-results topic

Now run the agent on the **planted** topic `obscure_topic`, whose search always returns `[]`. With
the buggy contract, the planner never gets a signal to stop paraphrasing — it climbs to the step
cap, exactly as in the lecture's case study. We keep the resulting tracer for diagnosis.

In [ ]:
tr_bad, outcome_bad = run_research_agent("obscure_topic", search_web)
print("planted run:", outcome_bad)
print("steps:", tr_bad.spans[0].attributes["steps"],
      "| total spans:", len(tr_bad.spans))

## Part D — Persist as JSONL, render the tree and the token account

A trace is only useful if it **outlives the run**. We serialise each span to one JSON object per
line (**JSONL**) — the format the pre-recorded `bad_run_trace.jsonl` already uses — then render two
views the lecture relies on: the **span tree** (read the *shape* first) and a **token/cost
account** (attribution per span kind).

Complete the gaps: write **one JSON object per line**, and reload by parsing **each line**.

In [ ]:
def save_trace_jsonl(tracer, path):
    """Persist a trace as JSONL: one span object per line."""
    with open(path, "w", encoding="utf-8") as f:
        for span in tracer.spans:
            f.write(json.dumps(asdict(span), ensure_ascii=False) + ___)   # GAP: newline per span


def load_trace_jsonl(path):
    """Reload a trace as a list of plain-dict spans."""
    spans = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                spans.append(___)                                          # GAP: parse the line
    return spans


os.makedirs("out", exist_ok=True)
save_trace_jsonl(tr_bad, "out/planted_run_trace.jsonl")
reloaded = load_trace_jsonl("out/planted_run_trace.jsonl")
print(f"Persisted and reloaded {len(reloaded)} spans.")
print("first span:", reloaded[0]["name"], "| kind:", reloaded[0]["kind"])

<details>
<summary><b>Click here for the solution</b></summary>

```python
f.write(json.dumps(asdict(span), ensure_ascii=False) + "\n")   # newline per span
...
spans.append(json.loads(line))                                 # parse the line
```

</details>

A **text-based tree renderer** (no matplotlib needed) that reads the *shape* at a
glance — the first step of the lecture's diagnosis method. Each span prints with its kind,
latency, and a one-line payload preview; children are indented under their parent.

In [ ]:
def render_tree(spans, max_children=6):
    """Print the span tree. Reads the SHAPE first (Slide 17, step 1)."""
    by_id = {s["span_id"]: s for s in spans}
    children = {}
    root = None
    for s in spans:
        pid = s["parent_id"]
        if pid is None:
            root = s
        children.setdefault(pid, []).append(s)

    def preview(s):
        a = s["attributes"]
        if s["kind"] == "tool":
            return f"args={a.get('args')} result={a.get('result')!r}"
        if s["kind"] == "llm":
            return f"in={a.get('input_tokens')}tok out={a.get('output_tokens')}tok"
        if s["kind"] in ("run",):
            return f"outcome={a.get('outcome')!r}"
        return ""

    def walk(s, depth):
        lat = s.get("end_ms")
        lat = f"{lat - s['start_ms']:6.1f}ms" if lat is not None else "  open "
        print(f"{'  ' * depth}{s['kind']:5} {s['name']:10} {lat}  {preview(s)}")
        kids = children.get(s["span_id"], [])
        for i, k in enumerate(sorted(kids, key=lambda x: x["start_ms"])):
            if i >= max_children:
                print(f"{'  ' * (depth + 1)}... ({len(kids) - max_children} more step subtrees)")
                break
            walk(k, depth + 1)

    walk(root, 0)


print("=== BAD RUN (first few step subtrees) ===")
render_tree(reloaded, max_children=4)

And the **token/cost account** — the lecture's per-span-kind attribution. We sum
input/output tokens over `llm` spans and apply a toy price to show cost attribution. Note how the
input tokens **grow every step**: the re-sent context is exactly what makes a looping run
expensive (and its traces large).

In [ ]:
PRICE_PER_1K = {"input": 0.0005, "output": 0.0015}   # toy prices, USD per 1k tokens


def token_account(spans):
    rows = []
    for s in spans:
        if s["kind"] == "llm":
            a = s["attributes"]
            rows.append({
                "span": s["name"],
                "input_tokens": a.get("input_tokens", 0),
                "output_tokens": a.get("output_tokens", 0),
            })
    df = pd.DataFrame(rows)
    df["cost_usd"] = (df["input_tokens"] / 1000 * PRICE_PER_1K["input"]
                      + df["output_tokens"] / 1000 * PRICE_PER_1K["output"])
    return df


acct = token_account(reloaded)
print(acct.to_string(index=False))
print(f"\nTOTAL: {acct['input_tokens'].sum()} input + {acct['output_tokens'].sum()} output tokens"
      f"  ->  ${acct['cost_usd'].sum():.4f}")
print("Notice: input tokens grow every step — the re-sent context dominates a looping run's cost.")

> **Q:** Why is recording full payloads — not just event counts and timings — non-negotiable for agent debugging?
<details><summary>Click for answer</summary>

Counts and timings can flag that something is anomalous (too many steps, too slow), but agent
failures are typically semantic: a judgment made on specific text. To debug a judgment you need
its exact input — the prompt as the model saw it — and its exact output. A summary of what the
model saw is interpretation, not evidence. Without payloads, the trace can locate *where* a run
went strange but never explain *why*.
</details>

> **Q:** Name the token categories worth accounting separately and explain why each matters.
<details><summary>Click for answer</summary>

Input tokens, because the growing context is re-sent on every loop step and dominates cost in long
runs; output tokens, usually priced higher per token; cached tokens, billed at a discount and key
to loop economics when prefixes repeat; and reasoning tokens for thinking models, which are paid
for but often invisible in the output. Separating them turns "this run was expensive" into an
attributable, actionable breakdown.
</details>

## Part E — Diagnose a prepared faulty run *from its trace*

Here is the professional scenario: a run failed in production; all you have is the recording
`data/bad_run_trace.jsonl`. **Diagnose it from the trace, not from the source code** (Slide 23's
rule). We load it, read the *shape*, *zoom* into the repeated block, and quantify the **loop
signature** — then you write the diagnosis as **Report task R1**.

First, load the recorded bad trace and read its shape.

In [ ]:
prepared = load_trace_jsonl("data/bad_run_trace.jsonl")
print(f"Loaded prepared trace: {len(prepared)} spans, "
      f"trace_id={prepared[0]['trace_id']}")
print("root outcome:", prepared[0]["attributes"]["outcome"])
print("\n=== SHAPE (step 1 of the diagnosis) ===")
render_tree(prepared, max_children=3)

**Step 2 — zoom.** Read the payloads of the repeated `search_web` spans. Because
we recorded them verbatim, the smoking gun is right there: every search returns the same value.

In [ ]:
tool_spans = [s for s in prepared if s["kind"] == "tool"]
print(f"{len(tool_spans)} tool spans. Their results:")
for s in tool_spans[:5]:
    print(f"  {s['name']}(args={s['attributes']['args']}) -> "
          f"result={s['attributes']['result']!r}  error={s['attributes']['error']!r}")
print("  ...")
distinct_results = {json.dumps(s["attributes"]["result"]) for s in tool_spans}
print(f"\nDistinct results across ALL {len(tool_spans)} searches:", distinct_results)
print("Every search returned the SAME empty list, reported as success (error=None).")

**Step 3 — quantify the loop signature.** A reusable detector for the longest run
of **consecutive tool calls that return the identical result**. The subtle point (which the
lecture flags): the model *paraphrases its query every step*, so the **arguments differ** each
time — what actually repeats is the **result** (every search returns the same empty list). Above a
small threshold, that repeated-result run *is* the loop signature, and it is the property we will
later assert in the regression test.

> **📝 Report task R4 (code)** appears in the cell below — complete `max_repeated_result`.

In [ ]:
def max_repeated_result(spans):
    """Loop-signature detector (Report task R4)."""
    # R4: return the length of the longest run of CONSECUTIVE tool spans that
    # share the same tool name AND the identical result. The model paraphrases
    # its query, so the ARGS differ every step — the repeated RESULT is what
    # betrays the loop. A value > threshold is the loop signature.
    ___


bad_sig = max_repeated_result(prepared)
ok_sig = max_repeated_result([asdict(s) for s in tr_ok.spans])
print(f"loop signature (longest identical-result run) — bad trace: {bad_sig}")
print(f"loop signature — healthy 'tracing' run:                    {ok_sig}")
THRESHOLD = 2
print(f"\nLoop detected in bad trace: {bad_sig > THRESHOLD}"
      f"  (threshold={THRESHOLD})")

> **📝 Report task R4 (code) — the loop-signature detector:** Complete the `max_repeated_result(spans)` gap in the cell below. It must return the **length of the longest run of consecutive tool spans that share the same tool name *and* the identical result** — the quantity that, above a small threshold, *is* the loop signature the lecture reads off the trace shape. Note the subtlety the lecture flags: the model **paraphrases its query every step**, so the *arguments* differ each time — comparing args would miss the loop entirely. What actually repeats is the **result** (every search returns the same empty list), so compare the tool name and the result (via canonical JSON so ordering does not matter). In your report, state the value it returns for the bad trace and for a healthy trace, and explain why the repeated **result** — not the arguments — is the reliable loop signal here.
> *No solution is provided — include your completed code and the two values in your lab report.*

> **📝 Report task R1 — read the trace, name the root cause (deliverable):** Using **only** the trace you rendered above (not the tool source code), write the lecture's compact deliverable: a **three-sentence diagnosis** of the nonterminating run. Follow the three-step method from the lecture — (1) *shape* (what does the tree's shape tell you?), (2) *zoom* (what do the repeated spans' payloads show?), (3) *root cause* (which tool, which result convention, and why does the model's response to it loop?). State explicitly whether the bug is in the model or in the tool contract, and justify it.
>
> *Three sentences is deliberate: if your diagnosis needs more, you have not yet found the root cause.*
> *No solution is provided — include your three-sentence diagnosis in your lab report.*

> **Q:** The root cause was a tool returning an empty list as a *successful* result. Explain why this is a tool-contract bug rather than a model bug.
<details><summary>Click for answer</summary>

Each model decision was locally reasonable: given a successful call with unhelpful output, retrying
with a rephrased query is sensible. The defect is in the interface semantics: the tool encoded "no
results exist" identically to ordinary success, giving the model no signal to change strategy
(S03 — tool results are the model's *perception*, and a result that looks like success while
meaning failure guarantees misinterpretation). The fix is in the **contract**: an explicit
no-results signal with guidance.
</details>

## Part F — Fix the contract, then prove it with a replay regression test

The fix is in the **tool contract**, not the model: `search_web_v2` signals "no results" as an
explicit `StopIteration` carrying guidance, instead of a silent empty list. The instrumented loop
already has a `StopIteration` branch that turns this into a clean, honest termination.

Then we build a **replay-based regression test** from the recorded bad trace. Replay re-runs the
loop while **serving tool results from the recording**, holding the environment fixed so only the
*decisions* vary — here we simply assert a **trajectory property** on the fixed run: *no identical
tool call repeats more than twice* (the loop-signature detector from Part E). Yesterday's diagnosed
failure becomes tonight's automated test.

Complete the gaps: raise the explicit no-results signal, and assert the trajectory property.

In [ ]:
def search_web_v2(query):
    """FIXED CONTRACT: 'no results' is an explicit signal with guidance, not a silent []."""
    topic = query.strip().lower().split()[0] if query.strip() else ""
    hits = PAGES_BY_TOPIC.get(topic, [])
    if not hits:
        # explicit no-results signal the loop can react to (guidance included)
        raise ___("no sources found; broaden the query or report the absence of sources")
    return [{"url": p["url"], "title": p["title"]} for p in hits]


def regression_test_no_loop(topic, search_tool, threshold=2):
    """Run the FIXED agent on the planted topic and assert it does NOT loop."""
    tr, outcome = run_research_agent(topic, search_tool)
    sig = max_repeated_result([asdict(s) for s in tr.spans])
    # the property that must hold: no identical-result tool call runs more than `threshold` times
    assert ___, f"loop signature {sig} exceeds threshold {threshold} — the bug is back!"
    return outcome, sig


# BEFORE the fix: the buggy tool loops to the step cap
tr_buggy, _ = run_research_agent("obscure_topic", search_web)
sig_before = max_repeated_result([asdict(s) for s in tr_buggy.spans])
print(f"buggy tool — loop signature: {sig_before}  (loops to the cap)")

# AFTER the fix: the regression test passes
outcome_fixed, sig_after = regression_test_no_loop("obscure_topic", search_web_v2)
print(f"fixed tool — outcome: {outcome_fixed}")
print(f"fixed tool — loop signature: {sig_after}  -> regression test PASSED")

<details>
<summary><b>Click here for the solution</b></summary>

```python
raise StopIteration("no sources found; broaden the query or report the absence of sources")
...
assert sig <= threshold, f"loop signature {sig} exceeds threshold {threshold} — the bug is back!"
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

`search_web_v2` repairs the **contract**: absence-of-results is now an explicit `StopIteration`
carrying *guidance*, so the loop's dormant `StopIteration` branch fires and the run terminates
honestly ("no sources found") instead of paraphrasing forever. `regression_test_no_loop` encodes
the diagnosed failure as a permanent **trajectory-property** assertion — `sig <= threshold` — using
the same loop-signature detector you completed in R4. If a future change reintroduces the silent
empty-list contract, `sig` climbs and the assertion fails *before a user suffers*. Note we assert a
*property* ("no identical call more than twice"), not an exact span sequence: many valid
trajectories exist for a nondeterministic agent, so exact-path matching would be brittle.
</details>

### Redact: a PII scrubber on the export path

Full-payload traces are a **liability** (GDPR): they capture user prompts, retrieved documents, and
secrets. The lecture's remedy is **redaction at ingestion — before storage**. We add a scrubber to
the export path and verify it with a **seeded email address**: the email is placed into a payload,
and we confirm it **never reaches the persisted file**.

Complete the gap: apply the scrubber to each span's serialised payload *before* writing it.

In [ ]:
PII_PATTERNS = {
    "email": re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+"),
    "api_key": re.compile(r"sk-[A-Za-z0-9]{8,}"),
}


def scrub(text):
    """Redact PII/secrets in a serialised span string (ingestion-time redaction)."""
    for name, pat in PII_PATTERNS.items():
        text = pat.sub(f"[REDACTED:{name}]", text)
    return text


def save_trace_jsonl_redacted(tracer, path):
    """Like save_trace_jsonl, but scrubs PII BEFORE anything reaches storage."""
    with open(path, "w", encoding="utf-8") as f:
        for span in tracer.spans:
            line = json.dumps(asdict(span), ensure_ascii=False)
            f.write(___ + "\n")          # GAP: scrub the line before writing it


# seed a run whose payload contains an email address, then export redacted
tr_seed = Tracer()
run = tr_seed.start("agent_run", "run", task="contact alice@example.com for the dataset")
fetch = tr_seed.start("fetch_page", "tool", args={"url": "https://example.com/x"})
fetch.set(result="Please email bob@secret.org and use key sk-ABCD1234EFGH", error=None)
tr_seed.end(fetch)
tr_seed.end(run)

save_trace_jsonl_redacted(tr_seed, "out/redacted_trace.jsonl")
raw = open("out/redacted_trace.jsonl", "r", encoding="utf-8").read()
print("email reached storage:", "alice@example.com" in raw or "bob@secret.org" in raw)
print("api key reached storage:", "sk-ABCD1234EFGH" in raw)
print("[REDACTED:email] present:", "[REDACTED:email]" in raw)
assert "alice@example.com" not in raw and "bob@secret.org" not in raw, "PII leaked to storage!"
print("OK — no seeded PII reached the persisted trace.")

<details>
<summary><b>Click here for the solution</b></summary>

```python
f.write(scrub(line) + "\n")   # scrub the line before writing it
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

Redaction happens **at ingestion**, on the export path, so the sensitive bytes *never reach the
file* — they cannot later be leaked by a breach, browsed by an over-privileged engineer, or
forgotten in a backup, which is exactly the data-minimisation posture the GDPR expects. Redacting
at *query* time instead would leave the raw data underneath, protected only by the query layer.
The cost is irreversibility: redacted payloads cannot be un-redacted for debugging, so the
redaction scope must be designed deliberately — regexes for direct identifiers (emails, keys) that
add little debugging value, while semantic content is kept.
</details>

> **Q:** Why must the tracing wrapper re-raise exceptions and record in a `finally` block?
<details><summary>Click for answer</summary>

Re-raising preserves behaviour: instrumentation must be observationally transparent, so the loop's
error handling sees exactly the exception it would have seen untraced — a recorder that changes
outcomes corrupts its own evidence. The `finally` block guarantees the span is timed, completed,
and exported on every path, including failures and cancellations; a span recorded only on success
is useless precisely on the runs you need to debug.
</details>

> **Q:** Why is PII redaction *at ingestion* preferable to redaction *at query time*?
<details><summary>Click for answer</summary>

Ingestion-time redaction means the sensitive data never reaches storage: it cannot be leaked by a
breach, browsed by an over-privileged engineer, or forgotten in a backup — and it directly
implements minimisation. Query-time redaction leaves raw data underneath, protected only by the
query layer, so every bypass (database access, exports, misconfiguration) exposes it. The cost is
irreversibility: redacted payloads cannot be un-redacted for debugging, so the redaction scope must
be designed deliberately.
</details>

> **Q (not exam-relevant):** What do the OpenTelemetry GenAI semantic conventions standardise, and what do they deliberately leave open?
<details><summary>Click for answer</summary>

They standardise *names and structures*: span types for inference, tool execution, and agent
operations, plus attribute names like `gen_ai.request.model` and `gen_ai.usage.input_tokens`, all
transportable over OTLP to any compliant backend. They leave **full payload capture opt-in** for
privacy and size reasons, and parts of the conventions remain experimental, so attribute names can
still shift between versions and should be pinned. (Our plain-Python `Span.attributes` dict plays
the role of these standard attributes.)
</details>

## Part G — Tuning & exploration (no gaps)

Things to play with — none of these cells contain gaps:

- **Change the loop threshold:** lower `THRESHOLD` in the detector and watch borderline runs flip
  between "loop" and "fine"; think about false positives on legitimately repetitive tasks.
- **Grow the context faster/slower:** edit the `context_tokens += 40` step in the loop and re-read
  the token account — see how re-sent context drives both cost and trace size.
- **Add a PII pattern:** extend `PII_PATTERNS` (phone numbers, IBANs) and re-verify the scrubber
  with a new seeded value.
- **Add another trajectory property:** e.g. assert the run terminates *under* the step budget, or
  that a `fetch_page` span exists before any "report drafted" outcome.
- **Optional end-to-end (needs Ollama):** let a real model be the planner for one step and record
  its actual prompt/completion/token counts as an `llm` span — the tracer does not care whether
  the planner is scripted or a real LLM.

In [ ]:
# Optional: record a REAL llm span from Ollama (the tracer is agnostic to the planner).
def trace_one_real_llm_call(topic="tracing"):
    if not OLLAMA_OK:
        print("Ollama not available — skipping the optional end-to-end cell.")
        return
    tr = Tracer()
    run = tr.start("agent_run", "run", task=f"decide a search query for {topic!r}")
    llm = tr.start("llm_call", "llm")
    prompt = (f"You are a research agent. Propose ONE short web-search query for the topic "
              f"{topic!r}. Answer with the query only.")
    resp = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}],
                       options={"temperature": 0.2})
    usage = resp.get("prompt_eval_count", 0), resp.get("eval_count", 0)
    llm.set(model=MODEL, prompt=prompt,
            completion=resp["message"]["content"].strip(),
            input_tokens=usage[0], output_tokens=usage[1])
    tr.end(llm)
    tr.end(run)
    print("Recorded a real llm span:")
    render_tree([asdict(s) for s in tr.spans])


trace_one_real_llm_call()

# interactive threshold explorer (falls back gracefully without ipywidgets)
try:
    from ipywidgets import interact, IntSlider

    def explore(threshold=2):
        loops = max_repeated_result(prepared)
        print(f"bad-trace loop signature = {loops}; "
              f"flagged as loop at threshold {threshold}: {loops > threshold}")

    interact(explore, threshold=IntSlider(2, 1, 25, 1))
except Exception:
    print("ipywidgets not available — call max_repeated_result(prepared) by hand.")

> **📝 Report task R2 — the cost of looking:** The lecture insists observability is *a feature you design, build, and pay for*. Using the token counts your tracer recorded (input tokens grow every step because the context is re-sent), (i) explain why a full-payload trace of an agent run is *megabytes, not log lines*, and (ii) state the lecture's **working rule** for sampling in development vs production, and why **tail-based** sampling suits agents better than **head-based** sampling. Then argue in 3–5 sentences whether your tracer should export spans **synchronously inside the loop** or **buffered and asynchronously**, and what a failing trace backend must never be allowed to do.
> *No solution is provided — include your answer in your lab report.*

> **📝 Report task R3 — replay, regression, and its limits:** You built a replay-based regression test that serves recorded tool results and asserts a **trajectory property** (no identical tool call repeated more than twice). In your report: (i) explain why a recorded trace makes a *nondeterministic* run into a *deterministic* test artifact, and exactly **where replay validity ends**; (ii) explain why **one passing replay proves little** for a stochastic agent and what the statistically honest alternative is (previewing S14); and (iii) give **two more** trajectory properties (not the one already asserted) that you would add for the research agent, and say why property checks beat exact-path matching.
> *No solution is provided — include your answer in your lab report.*

## Wrap-up

**Takeaways**

- **There is no stack trace for a bad decision.** Agent failures are semantic — nothing throws,
  and a rerun samples a new trajectory — so evidence must be *recorded at runtime or it is gone*.
- The **trace is the primary artifact**: a tree of timed, attributed **spans** per run, carrying
  full payloads, token/cost accounting, latency, and the parent–child links that encode causality.
- **One wrapper at the tool boundary** (`traced`) turns every call into a recorded span; record on
  every path (`finally`) and re-raise unchanged, so instrumentation is observationally transparent.
- **Read by shape first:** a uniform repeated block is a **loop signature**; the root cause here
  was a **tool contract** returning "no results" as success — not a model bug (S03).
- **Traces compound:** a recorded failure becomes a **replay regression test** (assert *trajectory
  properties*, not exact paths) and, eventually, an eval dataset — one passing replay proves little
  for a stochastic agent (sample many; S14).
- **Traces are a liability too:** redact PII **at ingestion**, before storage (GDPR); budget the
  storage, latency, and attention that observability itself costs.

**Next week (Session 12 — Multi-Agent Systems & Reflection):** several agents working together —
and reflecting on their own output. Today's trace is what will let you see *which* agent, in a
crowd of them, actually caused a failure.

**📝 For your lab report — checklist**

| # | Task | Where |
|---|------|-------|
| R1 | Three-sentence diagnosis of the nonterminating run, read from the trace (shape/zoom/root cause) | Part E |
| R2 | The cost of looking: trace size, dev/prod sampling rule, sync vs async export | after Part G |
| R3 | Replay & regression: determinism from recordings, where validity ends, statistical honesty, two more properties | after Part G |
| R4 | Code: complete `max_repeated_result` + report the bad vs healthy values | Part E |